In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 multiblock CPU summary recovery

This is a standard-library JSON recovery only. It reads the completed arm q records from the failed multiblock summary run and reconstructs the missing calibration statistics. It does not import PyTorch, load a model or tensor, read video, call FFmpeg, generate, decode, or encode. It writes a separate recovered summary and never changes the original result.

In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_multiblock_recovery_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


## Fixed recovery source

The source is fixed to `MyDrive/Video-WM/C2A_Multiblock_Diagnostic/c2a_multiblock_20260915T022411Z/result.json`. The output is a new recovered-summary directory. No dependency installation or CUDA runtime is required.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_multiblock_recovery_run.json'
RUN_ID = 'c2a_multiblock_recovered_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_Multiblock_Recovered_Summary') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
command = [sys.executable, '-m', 'runtime.c2a.recover_multiblock_summary', '--config', str(CONFIG), '--output', str(OUTPUT)]
subprocess.run(command, cwd=SOURCE, check=True)
print((OUTPUT / 'recovered_summary.json').read_text())


## Recovered result

The recovered summary recomputes only the fixed b/A, per-block inverse-calibrated state, equal aggregate, O/M and held-negative statistics from persisted q lists. It cannot repair or replace the original experiment, and it records no acceptance threshold.